# DDoS Intrusion Detection in IoT Networks Using Machine Learning

This notebook presents a cleaned and reproducible machine learning workflow for detecting possible DDoS/intrusion patterns in IoT network traffic data.

**Important note:** This is an early exploratory ML project. If the dataset contains a true label column, the notebook uses it for supervised learning. If no true label column is available, the notebook creates exploratory pseudo-labels using K-Means clustering and clearly treats the task as exploratory anomaly detection rather than fully validated DDoS classification.

## 1. Import Libraries and Set Configuration

The configuration cell below controls the dataset path, target column handling, feature-selection strategy, and reproducibility settings.

In [ ]:
# =============================
# 1. Imports and configuration
# =============================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
)

RANDOM_STATE = 42
TEST_SIZE = 0.30
TOP_K_FEATURES = 10

# Set this manually if you know the true target column, e.g. TARGET_COLUMN = "Label".
# Leave as None to allow automatic detection.
TARGET_COLUMN = None

# Candidate columns that may represent a class/label in common network datasets.
CANDIDATE_TARGET_COLUMNS = [
    "label", "Label", "class", "Class", "target", "Target", "attack", "Attack", "Inbound"
]

# Data path handling.
# This notebook is designed to work whether it is placed in the repository root or inside /notebooks.
possible_data_paths = [
    Path("data/iot_ddos_dataset.csv"),
    Path("../data/iot_ddos_dataset.csv"),
    Path("neww_data.csv"),
    Path("../neww_data.csv"),
]

DATA_PATH = next((p for p in possible_data_paths if p.exists()), None)

print("Configuration loaded.")
print(f"Random state: {RANDOM_STATE}")
print(f"Detected data path: {DATA_PATH}")

## 2. Load the Dataset

The cell below loads the CSV file and performs light column-name cleaning by removing leading/trailing spaces and replacing internal spaces with underscores.

In [ ]:
# =============================
# 2. Load dataset
# =============================

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place the CSV file at 'data/iot_ddos_dataset.csv' "
        "or update DATA_PATH manually in the configuration cell."
    )

df = pd.read_csv(DATA_PATH)

# Clean column names for easier downstream processing.
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.replace(" ", "_", regex=False)
)

print(f"Dataset shape: {df.shape}")
display(df.head())

## 3. Exploratory Data Inspection

This section checks the structure of the dataset, missing values, duplicated records, and potential target columns.

In [ ]:
# =============================
# 3. Basic inspection
# =============================

print("Dataset information:")
display(df.info())

print("
Summary statistics for numerical columns:")
display(df.describe().T)

print("
Missing values per column:")
missing_values = df.isna().sum().sort_values(ascending=False)
display(missing_values[missing_values > 0])

print(f"
Number of duplicated rows: {df.duplicated().sum()}")

print("
Columns in the dataset:")
print(list(df.columns))

In [ ]:
# Optional: inspect low-cardinality columns that may be labels/classes.
low_cardinality_summary = []
for col in df.columns:
    n_unique = df[col].nunique(dropna=True)
    if n_unique <= 20:
        low_cardinality_summary.append({
            "column": col,
            "n_unique": n_unique,
            "values": df[col].dropna().unique()[:10]
        })

low_cardinality_df = pd.DataFrame(low_cardinality_summary)
display(low_cardinality_df)

## 4. Prepare Features and Target

This notebook uses a true label column when available. If a true label column is not found, it creates pseudo-labels using K-Means clustering. This fallback is useful for exploratory anomaly detection, but it should not be interpreted as fully validated DDoS ground truth.

In [ ]:
# =============================
# 4. Target detection and dataset preparation
# =============================

working_df = df.copy()

# Drop duplicated rows for cleaner modelling.
working_df = working_df.drop_duplicates().reset_index(drop=True)

# Detect target column if not manually provided.
if TARGET_COLUMN is None:
    normalized_candidates = {c.lower(): c for c in working_df.columns}
    detected_target = None
    for candidate in CANDIDATE_TARGET_COLUMNS:
        if candidate.lower() in normalized_candidates:
            candidate_col = normalized_candidates[candidate.lower()]
            # Use candidate if it has a reasonable number of classes.
            if working_df[candidate_col].nunique(dropna=True) <= 20:
                detected_target = candidate_col
                break
    TARGET_COLUMN = detected_target

print(f"Target column selected: {TARGET_COLUMN}")

if TARGET_COLUMN is not None:
    # Supervised workflow using detected/provided target labels.
    y_raw = working_df[TARGET_COLUMN].copy()
    X_raw = working_df.drop(columns=[TARGET_COLUMN])
    label_source = "true_or_existing_label"

    # Encode target labels if they are not already numeric.
    if y_raw.dtype == "object" or str(y_raw.dtype).startswith("category"):
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y_raw), name="label")
        print("Target labels encoded as:")
        print(dict(zip(le.classes_, le.transform(le.classes_))))
    else:
        y = y_raw.astype(int).rename("label")

else:
    # Unsupervised fallback: no true labels available.
    # The pseudo-labels will be created after splitting/scaling to reduce test leakage.
    X_raw = working_df.copy()
    y = None
    label_source = "kmeans_pseudo_label"

# Keep only numeric features for this classical ML workflow.
X_raw = X_raw.select_dtypes(include=[np.number]).copy()

# Remove columns with no variation.
constant_cols = [col for col in X_raw.columns if X_raw[col].nunique(dropna=True) <= 1]
if constant_cols:
    X_raw = X_raw.drop(columns=constant_cols)
    print(f"Dropped constant columns: {constant_cols}")

# Basic missing-value handling: median imputation for numeric columns.
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

print(f"Feature matrix shape after cleaning: {X_raw.shape}")
print(f"Label source: {label_source}")

if y is not None:
    print("Class distribution:")
    display(y.value_counts())

## 5. Train-Test Split and Scaling

To avoid data leakage, scaling is fitted only on the training set and then applied to the test set.

In [ ]:
# =============================
# 5. Train-test split and scaling
# =============================

if y is not None:
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_raw,
        y,
        test_size=TEST_SIZE,
        stratify=y if y.nunique() > 1 else None,
        random_state=RANDOM_STATE,
    )
else:
    # For pseudo-label workflow, split the unlabeled data first.
    X_train_raw, X_test_raw = train_test_split(
        X_raw,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

scaler = MinMaxScaler()
X_train_scaled_array = scaler.fit_transform(X_train_raw)
X_test_scaled_array = scaler.transform(X_test_raw)

X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=X_raw.columns, index=X_train_raw.index)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_raw.columns, index=X_test_raw.index)

print(f"Training features: {X_train_scaled.shape}")
print(f"Test features: {X_test_scaled.shape}")

## 6. Optional Pseudo-Label Creation Using K-Means

This section runs only if no true target column was detected. K-Means is fitted on the training set only, then used to assign cluster-based pseudo-labels to the test set. The smallest training cluster is treated as the potential anomaly class.

In [ ]:
# =============================
# 6. K-Means pseudo-labeling if no target exists
# =============================

if y is None:
    kmeans = KMeans(n_clusters=3, n_init=15, random_state=RANDOM_STATE)
    train_clusters = kmeans.fit_predict(X_train_scaled)
    test_clusters = kmeans.predict(X_test_scaled)

    train_cluster_counts = pd.Series(train_clusters).value_counts()
    anomaly_cluster = train_cluster_counts.idxmin()

    y_train = pd.Series((train_clusters == anomaly_cluster).astype(int), index=X_train_scaled.index, name="label")
    y_test = pd.Series((test_clusters == anomaly_cluster).astype(int), index=X_test_scaled.index, name="label")

    print("No true label column was detected.")
    print("Pseudo-labels were generated using K-Means clustering.")
    print(f"Cluster treated as anomaly: {anomaly_cluster}")

print("Training label distribution:")
display(y_train.value_counts())

print("Test label distribution:")
display(y_test.value_counts())

## 7. Feature Selection

Feature selection is fitted on the training data only. This prevents the test set from influencing the selected features.

Two methods are used:

1. Mutual information
2. Random Forest feature importance

The final feature list uses the intersection of both methods when possible. If the intersection is too small, the notebook falls back to the union/top-ranked features.

In [ ]:
# =============================
# 7. Feature selection on training data only
# =============================

n_features_to_select = min(TOP_K_FEATURES, X_train_scaled.shape[1])

# Mutual information
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=X_train_scaled.columns).sort_values(ascending=False)
selected_mi = list(mi_series.head(n_features_to_select).index)

# Random Forest importance
rf_selector = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
rf_selector.fit(X_train_scaled, y_train)
rf_importance = pd.Series(rf_selector.feature_importances_, index=X_train_scaled.columns).sort_values(ascending=False)
selected_rf = list(rf_importance.head(n_features_to_select).index)

intersection_features = sorted(set(selected_mi).intersection(selected_rf))
union_features = list(dict.fromkeys(selected_mi + selected_rf))

# Use intersection if it gives enough features; otherwise use union/top features.
if len(intersection_features) >= 3:
    selected_features = intersection_features
    selection_strategy = "intersection_of_mutual_information_and_random_forest"
else:
    selected_features = union_features[:n_features_to_select]
    selection_strategy = "union_or_top_ranked_features"

print(f"Feature selection strategy: {selection_strategy}")
print(f"Number of selected features: {len(selected_features)}")
print("Selected features:")
for feature in selected_features:
    print(f"- {feature}")

feature_selection_table = pd.DataFrame({
    "mutual_information": mi_series,
    "random_forest_importance": rf_importance,
}).sort_values("mutual_information", ascending=False)

display(feature_selection_table.head(20))

In [ ]:
# Keep only selected features for modeling.
X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]

print(f"Selected training matrix: {X_train_selected.shape}")
print(f"Selected test matrix: {X_test_selected.shape}")

## 8. Model Training

Two classical machine learning models are trained:

1. Linear Support Vector Machine
2. AdaBoost with a linear SVM base estimator

In [ ]:
# =============================
# 8. Model training
# =============================

svm_model = SVC(
    kernel="linear",
    C=0.1,
    probability=True,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
svm_model.fit(X_train_selected, y_train)

# Compatibility helper for different scikit-learn versions.
base_svm = SVC(
    kernel="linear",
    C=0.1,
    probability=True,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

try:
    adaboost_svm_model = AdaBoostClassifier(
        estimator=base_svm,
        n_estimators=50,
        learning_rate=1.0,
        random_state=RANDOM_STATE,
    )
except TypeError:
    adaboost_svm_model = AdaBoostClassifier(
        base_estimator=base_svm,
        n_estimators=50,
        learning_rate=1.0,
        random_state=RANDOM_STATE,
    )

adaboost_svm_model.fit(X_train_selected, y_train)

print("Models trained successfully.")

## 9. Model Evaluation

The models are evaluated using accuracy, precision, recall, F1-score, classification report, confusion matrix, and ROC-AUC.

In [ ]:
# =============================
# 9. Evaluation helper function
# =============================

def evaluate_model(model, model_name, X_test, y_test):
    """Evaluate a classifier and return a dictionary of metrics."""
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        y_score = None

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score) if y_score is not None and len(np.unique(y_test)) == 2 else np.nan,
    }

    print(f"
{model_name} Performance")
    print("=" * (len(model_name) + 12))
    print(classification_report(y_test, y_pred, zero_division=0))

    return metrics, y_pred, y_score

svm_metrics, y_pred_svm, y_score_svm = evaluate_model(
    svm_model, "Linear SVM", X_test_selected, y_test
)

ada_metrics, y_pred_ada, y_score_ada = evaluate_model(
    adaboost_svm_model, "AdaBoost-SVM", X_test_selected, y_test
)

results_df = pd.DataFrame([svm_metrics, ada_metrics])
display(results_df)

In [ ]:
# =============================
# 9b. Confusion matrices
# =============================

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test, y_pred_svm, "Confusion Matrix: Linear SVM")
plot_confusion_matrix(y_test, y_pred_ada, "Confusion Matrix: AdaBoost-SVM")

In [ ]:
# =============================
# 9c. ROC curve
# =============================

plt.figure(figsize=(6, 5))

if y_score_svm is not None and len(np.unique(y_test)) == 2:
    fpr_svm, tpr_svm, _ = roc_curve(y_test, y_score_svm)
    roc_auc_svm = auc(fpr_svm, tpr_svm)
    plt.plot(fpr_svm, tpr_svm, label=f"Linear SVM (AUC = {roc_auc_svm:.3f})")

if y_score_ada is not None and len(np.unique(y_test)) == 2:
    fpr_ada, tpr_ada, _ = roc_curve(y_test, y_score_ada)
    roc_auc_ada = auc(fpr_ada, tpr_ada)
    plt.plot(fpr_ada, tpr_ada, label=f"AdaBoost-SVM (AUC = {roc_auc_ada:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 10. Summary and Interpretation

This notebook provides a cleaner and more reproducible version of the original exploratory ML workflow. The major improvements include:

- clearer project documentation
- fixed random state for reproducibility
- train-test splitting before scaling
- scaler fitted only on the training set
- feature selection fitted only on the training set
- cleaner model evaluation
- explicit handling of true labels versus K-Means pseudo-labels

### Important Limitation

If the notebook uses K-Means pseudo-labels, the results should be interpreted as exploratory anomaly-detection results, not as evidence of a fully validated DDoS classifier. A stronger future version should use a dataset with verified DDoS/benign ground-truth labels and external validation.